# Collate comparison chunks

The typing runtime was reclaimed after chunk 22 of 25. The finished chunks are on Drive; this rebuilds the combined table from them.

Run all cells. No Kleborate, no downloads.

## Cell 1 - Mount Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/phagematch-pk-comparison')
KLEB = PERSIST / 'kleborate'
print('exists:', KLEB.exists())
chunks = sorted(KLEB.glob('chunk_*'))
done  = sorted(KLEB.glob('chunk_*.done'))
print(f'{len(chunks)} chunk dirs, {len(done)} done markers')


## Cell 2 - Find the per-chunk Kleborate output

In [ ]:
import pandas as pd

# Kleborate writes a few TSV/TXT files per run; the main table is the
# one carrying a strain/assembly column. Probe rather than hardcode a
# filename, because the name differs between presets and versions.
cands = {}
for d in sorted(p for p in KLEB.glob('chunk_*') if p.is_dir()):
    for f in sorted(d.rglob('*')):
        if f.suffix.lower() in {'.txt', '.tsv'} and f.stat().st_size > 0:
            cands.setdefault(f.name, []).append(f)

for name, files in sorted(cands.items(), key=lambda kv: -len(kv[1])):
    print(f'{len(files):>3}  {name}')


## Cell 3 - Concatenate

Picks the file present in the most chunks that actually carries typing columns, so an AMR-only side table cannot be mistaken for the main one.

In [ ]:
import re

WANT = re.compile(r'K_locus|K_type|ST|species', re.I)

best, best_n = None, 0
for name, files in cands.items():
    try:
        head = pd.read_csv(files[0], sep='\t', nrows=1, dtype=str)
    except Exception:
        continue
    if any(WANT.search(c) for c in head.columns) and len(files) > best_n:
        best, best_n = name, len(files)

print('using:', best, f'({best_n} chunks)')

frames = []
for f in cands[best]:
    df = pd.read_csv(f, sep='\t', dtype=str)
    df['chunk'] = f.parent.name if f.parent.name.startswith('chunk') \
                  else f.parents[1].name
    frames.append(df)

kleb = pd.concat(frames, ignore_index=True)
print('combined:', kleb.shape)
print('chunks represented:', kleb['chunk'].nunique())
print(kleb.columns.tolist()[:20])


## Cell 4 - Sanity check before saving

A silent duplicate or a wave of unassigned K-loci would poison the comparison, so check here rather than discover it during analysis.

In [ ]:
col = next((c for c in kleb.columns if c.lower() in
            {'strain', 'assembly', 'name', 'input_file_name'}), None)
print('id column:', col)
if col:
    kleb['assembly'] = (kleb[col].astype(str)
                        .str.replace(r'\.(fna|fa|fasta)$', '', regex=True)
                        .str.strip())
    print('unique genomes:', kleb['assembly'].nunique(), 'of', len(kleb))

kcol = next((c for c in kleb.columns if 'K_locus' in c or c == 'K_type'), None)
print('K column:', kcol)
if kcol:
    print(kleb[kcol].value_counts().head(15))


## Cell 5 - Save and download

In [ ]:
kleb.to_csv('/content/kleborate_comparison_kp.csv', index=False)
kleb.to_csv(PERSIST / 'kleborate_comparison_kp.csv', index=False)
print('wrote kleborate_comparison_kp.csv', kleb.shape)

from google.colab import files
files.download('/content/kleborate_comparison_kp.csv')
